## Get all the imports

In [ ]:
import pytorch_lightning as pl
import torch
from torchvision import transforms
from torchvision.datasets import CIFAR10
from model import MyModel
from datacull.data import DCDataset
from datacull.methods.CCS import CCSDataLoader
from datacull.logger import DCLogger
from datacull.methods.CCS import AUMImportance

## Defining some global variables

In [ ]:
num_epochs = 20
batch_size = 128
pruning_rate = 0.8
beta = 0.1
num_strata = 50

## Create the data module class (CIFAR10 for simplicity)
- Since CCS is a static pruning technique, we need to first identify sample importance.
- For that, we build a normal pytorch lightning data module.
- The only difference is that the **pytorch dataset** is going to be **wrapped by the DPDataset** to enable indexing.

In [2]:
class PrePruningDataModule(pl.LightningDataModule):
    def __init__(self, batch_size, num_workers=20):
        self.batch_size = batch_size
        self.num_workers = num_workers
        # These are the transforms to be used if you want to train a model from scratch on the current dataset
        self.train_transform = transforms.Compose([
            transforms.RandomCrop(32, 4),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
        ])
        self.val_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
        ])
        super().__init__()
    
    
    def prepare_data(self, stage="None"):
        # This is the only line where we differ from normal code.
        # We wrap the CIFAR10 dataset (or your own dataset) with DPDataset to enable indexing
        self.train_set = DCDataset(CIFAR10(root='data/', train=True, download=True, transform=self.train_transform))
        self.val_set = DCDataset(CIFAR10(root='data/', train=False, download=True, transform=self.val_transform))
    
    
    def setup(self, stage: str):
        self.train_data_loader = torch.utils.data.DataLoader(self.train_set, batch_size=self.batch_size, shuffle=True, num_workers=self.num_workers)
        self.val_data_loader = torch.utils.data.DataLoader(self.val_set, batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers)
    
    
    def train_dataloader(self):
        return self.train_data_loader
    
    
    def val_dataloader(self):
        return self.val_data_loader

## Define the logger object
- This object logs a model's metrics such as predictions, loss, etc.

In [4]:
logger_object = DCLogger(trajectory_dir="model_trajectory/", save_every_k_epoch=1)

## Define a model for training
- We will use its training trajectory to find sample importance
- The logger object is used to log the metrics every iteration

In [5]:
model = MyModel(logger_object=logger_object)

## Create the data module object and train the model

In [ ]:
data_module = PrePruningDataModule(batch_size=batch_size, num_workers=4)
trainer = pl.Trainer(accelerator="gpu", devices=1, max_epochs=num_epochs, deterministic=True, enable_model_summary=True, num_sanity_val_steps=0)
trainer.fit(model, data_module)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
c:\ProgramData\anaconda3\Lib\site-packages\pytorch_lightning\trainer\connectors\logger_connector\logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
100%|██████████| 170M/170M [03:20<00:00, 852kB/s]  
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type               | Params | Mode 
-----

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

## Now compute the importance

In [ ]:
importance_object = AUMImportance(dataset=data_module.train_set, trajectory_length=5, logger_object=logger_object)
sample_importance = importance_object.compute_importance()

## Now, create a data module that will be used after pruning is done
- To make lives simpler, we will inherit the PrePruningDataModule to avoid multiple definitions of transforms, etc.
- This is the class where we use the CCSDataLoader to apply CCS pruning logic
- Since CCS is a static data pruning technique, the resample function ensures we sample once and then only shuffle the data every epoch

In [3]:
class PostPruningDataModule(PrePruningDataModule):
    def __init__(self, sample_importance, pruning_rate, beta, num_strata, batch_size, num_workers=20):
        self.sample_importance = sample_importance
        self.pruning_rate = 1 - pruning_rate
        self.beta = beta
        self.num_strata = num_strata
        super().__init__(batch_size, num_workers)
    
    def setup(self, stage: str):
        # Use the CCSDataLoader to apply CCS sampling strategy
        self.train_data_loader = CCSDataLoader(dataset=self.train_set, pruning_rate=self.pruning_rate, beta=self.beta, num_strata=self.num_strata, descending=False, batch_size=self.batch_size, num_workers=self.num_workers)
        self.val_data_loader = torch.utils.data.DataLoader(self.val_set, batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers)
    
    def train_dataloader(self):
        # Resample the training data loader based on sample importance
        self.train_data_loader.resample(self.sample_importance)
        return self.train_data_loader


In [ ]:
data_module = PostPruningDataModule(sample_importance=sample_importance, pruning_rate=pruning_rate, beta=beta, num_strata=num_strata, batch_size=batch_size)
model = MyModel()
# reload_dataloaders_every_n_epochs=True is important to ensure that the dataloader is reloaded every epoch to reflect the new sampling.
# For static methods, this only shuffles the data, but for dynamic methods, this is necessary to update the sampling based on new importance scores.
# This is a design choice to unify both static and dynamic methods under the same flag.
trainer = pl.Trainer(accelerator="gpu", devices=1, max_epochs=num_epochs, deterministic=True, enable_model_summary=False, num_sanity_val_steps=0, reload_dataloaders_every_n_epochs=True)
trainer.fit(model, data_module)